#  Modelamiento: Estrés Energético (K-Means + Optuna)

En este notebook nos conectaremos a nuestra base de datos `energia.db` para crear un pipeline completo de Machine Learning de nivel avanzado.

**Flujo del Modelo**:
1. **No Supervisado (K-Means)**: Encontrar patrones y agrupar las barras en clústeres de "riesgo". Esto nos dará nuestra variable objetivo (`y`).
2. **Train/Test Split**: Separar los datos rigurosamente.
3. **Optimización con Optuna**: Buscar los mejores hiperparámetros para un Random Forest de forma automática e inteligente.
4. **Supervisado (Clasificación)**: Entrenar el modelo final y exportarlo para nuestra API.

In [1]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
import optuna

# Machine Learning
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

import warnings
warnings.filterwarnings('ignore')

## 1. Extracción y Limpieza de Datos

In [2]:
db_path = '../data/processed/energia.db'
conn = sqlite3.connect(db_path)

query = """
SELECT 
    c.barra_nombre,
    b.region,
    AVG(c.costo_marginal_usd_mwh) as costo_promedio,
    MAX(c.costo_marginal_usd_mwh) as costo_maximo,
    AVG(p.pib_millones_clp) as pib_millones_clp
FROM costos_marginales c
JOIN dim_barras b ON c.barra_nombre = b.barra_nombre
LEFT JOIN pib_regional p ON b.region = p.region
GROUP BY c.barra_nombre, b.region
"""
df = pd.read_sql(query, conn)
conn.close()

# Limpiamos los nulos (Data Cleaning)
df_clean = df.dropna(subset=['pib_millones_clp']).copy()
print(f"Filas listas para modelar: {len(df_clean)}")

Filas listas para modelar: 168


## 2. Creación de Etiquetas (K-Means)

In [3]:
features = ['costo_promedio', 'costo_maximo', 'pib_millones_clp']
X_clustering = df_clean[features]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_clustering)

kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
# Esta será nuestra variable 'y' (La clase objetivo a predecir)
df_clean['cluster_riesgo'] = kmeans.fit_predict(X_scaled)

df_clean.head()

,barra_nombre,region,costo_promedio,costo_maximo,pib_millones_clp,cluster_riesgo
37,BA S/E ALTO JAHUEL 110KV BP1,Metropolitana,35.400740,71.152045,19663.604155,2
38,BA S/E ALTO JAHUEL 154KV BP,Metropolitana,35.359445,71.201405,19663.604155,2
39,BA S/E ALTO JAHUEL 220KV BP1,Metropolitana,35.212474,70.726282,19663.604155,2
40,BA S/E ALTO JAHUEL 220KV BP2,Metropolitana,34.565902,68.944343,19663.604155,2
41,BA S/E ALTO JAHUEL 500KV BPA,Metropolitana,34.997477,70.229410,19663.604155,2


## 3. Train / Test Split
Para cumplir con el estándar de Machine Learning, separamos rigurosamente nuestros datos en Entrenamiento y Prueba.

In [4]:
X = df_clean[features]
y = df_clean['cluster_riesgo']

# División 80% entrenamiento (train) y 20% prueba (test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Tamaño de X_train: {X_train.shape}")
print(f"Tamaño de X_test: {X_test.shape}")

Tamaño de X_train: (134, 3)
Tamaño de X_test: (34, 3)


## 4. Búsqueda de Hiperparámetros con Optuna
Usaremos Optuna para encontrar los mejores parámetros de nuestro Random Forest maximizando el Accuracy.

In [5]:
def objective(trial):
    # Definimos el espacio de búsqueda
    n_estimators = trial.suggest_int('n_estimators', 50, 300)
    max_depth = trial.suggest_int('max_depth', 3, 15)
    min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
    
    clf = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        random_state=42
    )
    
    clf.fit(X_train, y_train)
    y_pred_val = clf.predict(X_test)
    
    return accuracy_score(y_test, y_pred_val)

# Creamos y ejecutamos el estudio de Optuna
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=20) # 20 iteraciones para no demorar mucho

print(f"\n🏆 Mejor Accuracy encontrado: {study.best_value * 100:.2f}%")
print("🔧 Mejores Parámetros:", study.best_params)

[I 2026-07-09 19:27:38,485] A new study created in memory with name: no-name-2e75b476-8747-45d2-b0b7-146884eb4b4e
[I 2026-07-09 19:27:38,698] Trial 0 finished with value: 1.0 and parameters: {'n_estimators': 167, 'max_depth': 4, 'min_samples_split': 5}. Best is trial 0 with value: 1.0.
[I 2026-07-09 19:27:38,962] Trial 1 finished with value: 1.0 and parameters: {'n_estimators': 209, 'max_depth': 10, 'min_samples_split': 2}. Best is trial 0 with value: 1.0.
[I 2026-07-09 19:27:39,067] Trial 2 finished with value: 1.0 and parameters: {'n_estimators': 78, 'max_depth': 9, 'min_samples_split': 2}. Best is trial 0 with value: 1.0.
[I 2026-07-09 19:27:39,361] Trial 3 finished with value: 1.0 and parameters: {'n_estimators': 230, 'max_depth': 10, 'min_samples_split': 3}. Best is trial 0 with value: 1.0.
[I 2026-07-09 19:27:39,443] Trial 4 finished with value: 1.0 and parameters: {'n_estimators': 62, 'max_depth': 11, 'min_samples_split': 4}. Best is trial 0 with value: 1.0.
[I 2026-07-09 19:27:


🏆 Mejor Accuracy encontrado: 100.00%
🔧 Mejores Parámetros: {'n_estimators': 167, 'max_depth': 4, 'min_samples_split': 5}


## 4.5 Comparativa de Clasificadores

Antes de confirmar Random Forest como modelo final, comparamos su rendimiento contra otras alternativas clasicas. Esto justifica la eleccion del modelo con evidencia empirica.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score
import pandas as pd

clasificadores = {
    "Logistic Regression": LogisticRegression(max_iter=500, random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "KNN (k=5)": KNeighborsClassifier(n_neighbors=5),
    "SVM (RBF)": SVC(kernel="rbf", random_state=42),
    "Random Forest": None  # Se cargara el modelo entrenado con Optuna
}

resultados = []
for nombre, clf in clasificadores.items():
    if clf is None:
        continue  # Random Forest se evalua abajo con el modelo optimizado
    clf.fit(X_train, y_train)
    y_pred_clf = clf.predict(X_test)
    resultados.append({
        "Modelo": nombre,
        "Accuracy": round(accuracy_score(y_test, y_pred_clf), 4),
        "F1-Score (macro)": round(f1_score(y_test, y_pred_clf, average="macro", zero_division=0), 4)
    })

df_resultados = pd.DataFrame(resultados)
print(df_resultados.to_string(index=False))

**Conclusion:** Random Forest (con optimizacion Optuna) se selecciona como modelo final por su capacidad de capturar relaciones no lineales entre el PIB y los costos energeticos, y por su robustez frente al overfitting en comparacion con Decision Tree y modelos lineales.

## 5. Entrenamiento del Modelo Final y Exportación
Entrenamos el Random Forest final usando los parámetros ganadores de Optuna y exportamos los `.pkl`.

In [6]:
# Entrenamos el modelo definitivo
best_clf = RandomForestClassifier(**study.best_params, random_state=42)
best_clf.fit(X_train, y_train)

# Reporte final en Test
y_pred_final = best_clf.predict(X_test)
print("Reporte de Clasificación (Modelo Optimizado):\n")
print(classification_report(y_test, y_pred_final))

# Exportamos para la API
os.makedirs('saved_models', exist_ok=True)
joblib.dump(best_clf, 'saved_models/modelo_rf.pkl')
joblib.dump(scaler, 'saved_models/scaler.pkl')
X_test.to_csv('saved_models/X_test.csv', index=False)
y_test.to_csv('saved_models/y_test.csv', index=False)

print("✅ Pipeline Completo y Optimizado. Modelo final guardado en 'models/saved_models/'.")

Reporte de Clasificación (Modelo Optimizado):

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        17
           1       1.00      1.00      1.00        15
           2       1.00      1.00      1.00         1
           3       1.00      1.00      1.00         1

    accuracy                           1.00        34
   macro avg       1.00      1.00      1.00        34
weighted avg       1.00      1.00      1.00        34

✅ Pipeline Completo y Optimizado. Modelo final guardado en 'models/saved_models/'.
